<div style="background-color:#000;"><img src="pqn.png"></img></div><div><a href="https://pyquantnews.com/">PyQuant News</a> is where finance practitioners level up with Python for quant finance, algorithmic trading, and market data analysis. Looking to get started? Check out the fastest growing, top-selling course to <a href="https://www.pyquantnews.com/getting-started-with-python-for-quant-finance/">get started with Python for quant finance</a>. For educational purposes. Not investment advice. Use at your own risk.</div>

## Library installation

This installs the libraries needed to fetch the price data, run the clustering, and draw the charts. Run it once at the top of the notebook.

In [ ]:
!pip install pandas matplotlib scikit-learn lxml yfinance

Install openbb_terminal yourself, in its own virtual environment, before you open the notebook. It pins its own versions of pandas and scikit-learn and will downgrade whatever pip already put there, so restart the kernel once it finishes. lxml is in the command above because pandas needs it to read the Dow membership table off Wikipedia.

## Imports and setup

We use pandas to hold the price table and compute daily changes, matplotlib for the two charts, KMeans from scikit-learn to sort the 30 stocks into groups, sqrt from math to put volatility on a yearly scale, and the OpenBB SDK to download the closes.

In [ ]:
from math import sqrt
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

In [ ]:
import yfinance as yf

Stocks with nearly identical return and volatility still print on top of each other, so widen the figure when that happens. Import KMeans from sklearn.cluster and not from sklearn itself, or you get ImportError: cannot import name 'KMeans'.

## Pull Dow tickers and daily prices

Read the Dow Jones Industrial Average components straight from Wikipedia so the ticker list reflects current membership instead of something typed by hand.

In [ ]:
symbols = [
    "AAPL", "AMGN", "AMZN", "AXP", "BA", 
    "CAT", "CRM", "CSCO", "CVX", "DIS", 
    "DOW", "GOOGL", "GS", "HD", "HON", 
    "IBM", "JNJ", "JPM", "KO", "MCD", 
    "MMM", "MRK", "MSFT", "NKE", "NVDA", 
    "PG", "SHW", "TRV", "UNH", "V", "WMT"
]


Download daily closes for all 30 tickers in one call, covering January 2020 through December 2022, then print data.shape to see what actually came back.

In [ ]:
data = yf.download(
    symbols,
    start="2020-01-01",
    end="2022-12-31",
).Close

Grab the historical data using yFinance.

## Reduce each stock to two numbers

Collapse each price series into an average yearly return and a yearly volatility, the standard deviation of daily percentage returns scaled up to a year, with missing days skipped by describe.

In [ ]:
moments = (
    data
    .pct_change()
    .describe()
    .T[["mean", "std"]]
    .rename(columns={"mean": "returns", "std": "vol"})
) * [252, sqrt(252)]

Order gets thrown out here. describe reduces each return column to a mean and a standard deviation, then the mean is multiplied by 252 for the trading days in a year and the standard deviation by the square root of 252, since variance adds over time rather than the deviation itself. Two stocks with the same average and the same day-to-day variability sit in the same spot on the chart even if one of them earned its whole return in a single month.

## Group the stocks with KMeans

Fit KMeans on the two unscaled columns for every group count from 2 through 14 and record how tightly the stocks sit around their group centers.

In [ ]:
sse = []
for k in range(2, 15):
    kmeans = KMeans(n_clusters=k, n_init=10)
    kmeans.fit(moments)
    sse.append(kmeans.inertia_)
plt.plot(range(2, 15), sse)
plt.title("Elbow Curve")
plt.show()

Both features are yearly decimals of similar size, so I fit them unscaled. inertia_ is the sum of squared distances from each stock to its own center and it falls every time you add a group, so the lowest value at k=14 tells you nothing. Print the sse list, find where the drop from one k to the next flattens, then fit that k and the ones either side of it, because with 30 stocks the bend is rarely sharp.

Refit with five groups, where the drop in sse flattens on this curve, and keep the group number assigned to each stock.

In [ ]:
kmeans = KMeans(n_clusters=5, n_init=10).fit(moments)

n_init=10 runs the fit ten times from different random starting centers and keeps the lowest inertia, since one start can settle into a local minimum that splits a dense cloud in half and merges two others. I never set random_state here, so membership usually holds across runs while the group numbers get shuffled. Row i of kmeans.labels_ belongs to row i of moments, and that number is an arbitrary tag rather than a rank.

Plot each stock by return and volatility, color the point by its group, and write the ticker and group number beside it.

In [ ]:
plt.scatter(
    moments.returns,
    moments.vol,
    c=kmeans.labels_,
    cmap="rainbow",
)
plt.title("Dow Jones stocks by return and volatility (K=5)")
for i in range(len(moments.index)):
    txt = f"{moments.index[i]} ({kmeans.labels_[i]})"
    xy = tuple(moments.iloc[i, :] + [0, 0.01])
    plt.annotate(txt, xy)
plt.show()

Without the annotations you cannot tell which dot is which, and reading off which tickers landed in the same group is the whole point. Run pd.Series(kmeans.labels_).value_counts() to get the size of each group. The crowded group holds stocks with similar yearly return and similar day-to-day variability, which is all this measured, so check their correlations before you treat five names from it as five separate positions.

<a href="https://pyquantnews.com/">PyQuant News</a> is where finance practitioners level up with Python for quant finance, algorithmic trading, and market data analysis. Looking to get started? Check out the fastest growing, top-selling course to <a href="https://www.pyquantnews.com/getting-started-with-python-for-quant-finance/">get started with Python for quant finance</a>. For educational purposes. Not investment advice. Use at your own risk.